In [1]:
%pip install transformers datasets accelerate torch scikit-learn

     ---------------------------------------- 12.1/12.1 MB 3.5 MB/s eta 0:00:00
     -------------------------------------- 559.1/559.1 kB 5.0 MB/s eta 0:00:00
     -------------------------------------- 389.2/389.2 kB 4.0 MB/s eta 0:00:00
     -------------------------------------- 124.1/124.1 MB 4.3 MB/s eta 0:00:00
     -------------------------------------- 796.8/796.8 kB 6.3 MB/s eta 0:00:00
     ---------------------------------------- 2.9/2.9 MB 9.6 MB/s eta 0:00:00
     ---------------------------------------- 123.1/123.1 kB ? eta 0:00:00
     -------------------------------------- 355.5/355.5 kB 7.3 MB/s eta 0:00:00
     ---------------------------------------- 100.0/100.0 kB ? eta 0:00:00
     ---------------------------------------- 120.0/120.0 kB ? eta 0:00:00
     ---------------------------------------- 144.5/144.5 kB ? eta 0:00:00
     ------------------------------------- 203.9/203.9 kB 12.1 MB/s eta 0:00:00
     -------------------------------------- 818.2/818.2 kB 8.6


[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.14.0+cpu
GPU available: False


In [3]:
df = pd.read_csv("../data/processed/truthlens_dataset.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset shape: (39103, 4)

Columns:
['title', 'text', 'label', 'content']


,title,text,label,content
0,Many 'lost' voters say they have found their c...,WASHINGTON/NEW YORK (Reuters) - Ted Wade hasn’...,1,Many 'lost' voters say they have found their c...
1,"Iranian Revolutionary Guards commander, anothe...",BEIRUT (Reuters) - A commander in Iran s elite...,1,"Iranian Revolutionary Guards commander, anothe..."
2,Cambodian leader gets China's backing as West ...,PHNOM PENH (Reuters) - Cambodian Prime Ministe...,1,Cambodian leader gets China's backing as West ...
3,Ex-KKK Grand Wizard: Maine Gov. Right To Call...,David Duke is perhaps one of the most notoriou...,0,Ex-KKK Grand Wizard: Maine Gov. Right To Call ...
4,FINGER-WAGGING GRANNY HILLARY CLINTON Blows Up...,Hillary Clinton shows her true colors here wit...,0,FINGER-WAGGING GRANNY HILLARY CLINTON Blows Up...


In [4]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Training: 31282
Validation: 3910
Test: 3911


In [5]:
print("\nTraining distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

print("\nTest distribution:")
print(test_df["label"].value_counts())


Training distribution:
label
1    16957
0    14325
Name: count, dtype: int64

Validation distribution:
label
1    2119
0    1791
Name: count, dtype: int64

Test distribution:
label
1    2120
0    1791
Name: count, dtype: int64


In [6]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["content", "label"]].reset_index(drop=True)
)

val_dataset = Dataset.from_pandas(
    val_df[["content", "label"]].reset_index(drop=True)
)

test_dataset = Dataset.from_pandas(
    test_df[["content", "label"]].reset_index(drop=True)
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

c:\Users\LENOVO\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['content', 'label'],
    num_rows: 31282
})
Dataset({
    features: ['content', 'label'],
    num_rows: 3910
})
Dataset({
    features: ['content', 'label'],
    num_rows: 3911
})


In [7]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded!")

Tokenizer loaded!


In [8]:
def tokenize_function(batch):
    return tokenizer(
        batch["content"],
        truncation=True,
        max_length=512
    )

In [9]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

Map: 100%|██████████| 3911/3911 [00:02<00:00, 1395.63 examples/s]


In [10]:
print(tokenized_train[0])

{'content': 'Watch Donald Trump Fail So Hard At Trying To Pander To Right-Wing Christians (VIDEO) During an interview on CNN s State of the Union on Sunday, Donald Trump was questioned by host, Jake Tapper, on his religious beliefs and support among the socially conservative, right-wing Christians.Things have been getting heated between Republican primary frontrunner Donald Trump, and the runner-up Ted Cruz, to say the least. Cruz has been surging in the polls as Trump maintains his lead. While Trump s lead appears to be unshakable, his support is also static. So the two have been at each other s throats trying to win over voters. In order to win over those voters, they have to convince people that they are the true spokesman for god and the Christian religion.In that arena, Cruz has a major advantage over Trump.  During the Family Leadership Summit, last July, Trump made the mistake of saying something somewhat reasonable to a crowd of hardcore social conservatives. He was asked if he

In [13]:
print(train_dataset)
print(tokenized_train[0])

Dataset({
    features: ['content', 'label'],
    num_rows: 31282
})
{'label': 0, 'input_ids': [101, 3422, 6221, 8398, 8246, 2061, 2524, 2012, 2667, 2000, 6090, 4063, 2000, 2157, 1011, 3358, 8135, 1006, 2678, 1007, 2076, 2019, 4357, 2006, 13229, 1055, 2110, 1997, 1996, 2586, 2006, 4465, 1010, 6221, 8398, 2001, 8781, 2011, 3677, 1010, 5180, 11112, 4842, 1010, 2006, 2010, 3412, 9029, 1998, 2490, 2426, 1996, 14286, 4603, 1010, 2157, 1011, 3358, 8135, 1012, 2477, 2031, 2042, 2893, 9685, 2090, 3951, 3078, 2392, 23195, 6221, 8398, 1010, 1998, 1996, 5479, 1011, 2039, 6945, 8096, 1010, 2000, 2360, 1996, 2560, 1012, 8096, 2038, 2042, 7505, 4726, 1999, 1996, 14592, 2004, 8398, 9319, 2010, 2599, 1012, 2096, 8398, 1055, 2599, 3544, 2000, 2022, 4895, 7377, 2912, 3468, 1010, 2010, 2490, 2003, 2036, 10763, 1012, 2061, 1996, 2048, 2031, 2042, 2012, 2169, 2060, 1055, 3759, 2015, 2667, 2000, 2663, 2058, 7206, 1012, 1999, 2344, 2000, 2663, 2058, 2216, 7206, 1010, 2027, 2031, 2000, 8054, 2111, 2008, 2027,

In [12]:
tokenized_train = tokenized_train.remove_columns(["content"])
tokenized_val = tokenized_val.remove_columns(["content"])
tokenized_test = tokenized_test.remove_columns(["content"])

print(tokenized_train)

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 31282
})


In [14]:
from transformers import AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("DistilBERT model loaded!")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7715.08it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT model loaded!


In [15]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [16]:
train_small = tokenized_train.select(range(2000))
val_small = tokenized_val.select(range(500))

print("Small training set:", len(train_small))
print("Small validation set:", len(val_small))

Small training set: 2000
Small validation set: 500


In [17]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="../models/distilbert_test",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none"
)

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_small,
    eval_dataset=val_small,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer created successfully!")

Trainer created successfully!


In [19]:
train_result = trainer.train()

print("Training completed!")

c:\Users\LENOVO\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.021242,0.002541,0.998000,1.000000,0.996364,0.998179


Training completed!


In [20]:
eval_result = trainer.evaluate()

print(eval_result)

c:\Users\LENOVO\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.021242,0.002541,1,0.998000,1.000000,0.996364,0.998179


{'eval_loss': 0.0025413057301193476, 'eval_accuracy': 0.998, 'eval_precision': 1.0, 'eval_recall': 0.9963636363636363, 'eval_f1': 0.9981785063752276}


In [21]:
test_result = trainer.predict(tokenized_test)

print("Test Results:")
print(test_result.metrics)

c:\Users\LENOVO\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Test Results:
{'test_loss': 0.006078030914068222, 'test_accuracy': 0.9989772436716953, 'test_precision': 0.9985862393967955, 'test_recall': 0.9995283018867924, 'test_f1': 0.9990570485619991, 'test_runtime': 1851.7347, 'test_samples_per_second': 2.112, 'test_steps_per_second': 0.528}


In [22]:
from transformers import TrainingArguments, Trainer

final_training_args = TrainingArguments(
    output_dir="../models/distilbert_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    save_total_limit=1,
    report_to="none"
)

final_trainer = Trainer(
    model=model,
    args=final_training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Final Trainer created successfully!")

Final Trainer created successfully!


In [24]:
def tokenize_function_256(batch):
    return tokenizer(
        batch["content"],
        truncation=True,
        max_length=256
    )

tokenized_train_256 = train_dataset.map(
    tokenize_function_256,
    batched=True
)

tokenized_val_256 = val_dataset.map(
    tokenize_function_256,
    batched=True
)

tokenized_test_256 = test_dataset.map(
    tokenize_function_256,
    batched=True
)

print("Tokenization completed!")

Map: 100%|██████████| 3911/3911 [00:02<00:00, 1773.74 examples/s]

Tokenization completed!


In [25]:
tokenized_train_256 = tokenized_train_256.remove_columns(["content"])
tokenized_val_256 = tokenized_val_256.remove_columns(["content"])
tokenized_test_256 = tokenized_test_256.remove_columns(["content"])

print(tokenized_train_256)
print(tokenized_val_256)
print(tokenized_test_256)

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 31282
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3910
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3911
})


In [27]:
print("Train columns:", tokenized_train_256.column_names)
print("Validation columns:", tokenized_val_256.column_names)
print("Test columns:", tokenized_test_256.column_names)

print("\nRows:")
print("Train:", len(tokenized_train_256))
print("Validation:", len(tokenized_val_256))
print("Test:", len(tokenized_test_256))

Train columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
Validation columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
Test columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']

Rows:
Train: 31282
Validation: 3910
Test: 3911


In [28]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Create a fresh DistilBERT model
model_256 = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

final_training_args_256 = TrainingArguments(
    output_dir="../models/distilbert_final_256",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    save_total_limit=1,
    report_to="none"
)

final_trainer_256 = Trainer(
    model=model_256,
    args=final_training_args_256,
    train_dataset=tokenized_train_256,
    eval_dataset=tokenized_val_256,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("256-token Trainer created successfully!")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2991.79it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


256-token Trainer created successfully!


In [29]:
print("Starting full DistilBERT 256-token training...")

final_train_result_256 = final_trainer_256.train()

print("Full 256-token training completed!")


Starting full DistilBERT 256-token training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000034,0.008468,0.998721,0.999056,0.998584,0.998820


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.37it/s]


Full 256-token training completed!


In [30]:
final_test_result_256 = final_trainer_256.predict(tokenized_test_256)

print("Final DistilBERT Test Results:")
print(final_test_result_256.metrics)

c:\Users\LENOVO\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Final DistilBERT Test Results:
{'test_loss': 0.0005111440550535917, 'test_accuracy': 0.9997443109179238, 'test_precision': 1.0, 'test_recall': 0.9995283018867924, 'test_f1': 0.9997640953054966, 'test_runtime': 1511.3744, 'test_samples_per_second': 2.588, 'test_steps_per_second': 0.647}


In [31]:
import os

model_path = "../models/distilbert_final_256"

print("Model files:")

for file in os.listdir(model_path):
    print(file)

Model files:
checkpoint-7821


In [32]:
import os

checkpoint_path = "../models/distilbert_final_256/checkpoint-7821"

print("Checkpoint files:")

for file in os.listdir(checkpoint_path):
    print(file)

Checkpoint files:
config.json
model.safetensors
optimizer.pt
rng_state.pth
scheduler.pt
tokenizer.json
tokenizer_config.json
trainer_state.json
training_args.bin
